# Simulating Image Modalities
This tutorial will demonstrate how to simulate a bacterium with different image modalities as well as how to generate a movie where this particle diffuses with active Brownian motion.

In [ ]:
import deeptrack as dt
import numpy as np
from matplotlib import pyplot as plt
from IPython.display import HTML
from matplotlib.animation import FuncAnimation

## Defining a bacterium scatterer
We first define a cylidrical volume to act as a bacterium using the DeepTrack2 `Scatterer` class.

In [ ]:
from deeptrack.scatterers import Scatterer
from deeptrack.types import PropertyLike, ArrayLike
from typing import Dict, Union

class Rod(Scatterer):
    """Generates a rod scatterer.

    Parameters
    ----------
    radius: float or array_like [float (, float)]
        Defines the half‑dimensions of the rod. The first value is the half‑length
        (L) along the rod's long axis and the second is the half‑thickness (T)
        in the center. If only one value is provided, a square rod is assumed.
        
    rotation: float
        Orientation angle of the rod in the imaging plane in radians.
        
    position: array_like[float, float (, float)]
        The position of the rod. A third index is optional and represents the
        position in the direction normal to the camera plane.
        
    z: float
        The position in the direction normal to the camera plane. Used if `position`
        is of length 2.
        
    value: float
        A default value of the characteristic of the particle.
        
    upsample: int
        Upsamples the calculations of the pixel occupancy fraction.
        
    transpose: bool
        If True, the rod is transposed so that the first axis of the size (radius)
        aligns with the first axis of the created volume. This is applied before rotation.
    """


    def __init__(
        self,
        radius: PropertyLike[float] = 1e-6,
        rotation: PropertyLike[float] = 0,
        transpose: PropertyLike[bool] = False,
        **kwargs,
    ) -> None:
        super().__init__(radius=radius, rotation=rotation, transpose=transpose, **kwargs)

    def _process_properties(
        self,
        properties: Dict
    ) -> Dict:
        
        """Ensure the rod's half‑dimensions (radius) is an array of length 2.
        If a single value is provided, a square rod is assumed.
        """
        properties = super()._process_properties(properties)
        radius = np.array(properties["radius"])
        if radius.ndim == 0:
            radius = np.array((properties["radius"], properties["radius"]))
        elif radius.size == 1:
            radius = np.array((*radius,) * 2)
        else:
            radius = radius[:2]
        properties["radius"] = radius
        return properties

    def get(
        self,
        *ignore,
        radius: Union[ArrayLike[float], float],
        rotation: PropertyLike[float],
        voxel_size: PropertyLike[float],
        transpose: PropertyLike[bool],
        **kwargs
    ) -> ArrayLike[float]:

        # If not transposed, swap to align the half‑dimensions with grid axes.
        if not transpose:
            radius = radius[::-1]

        # Rod parameters.
        L = radius[0]   # half‑length along the rod.
        T = radius[1]   # central half‑thickness.

        # Define taper parameters.
        taper_ratio = 0.05      # fraction of L used for tapering.
        taper_factor = 0.5      # at the rod’s tip, thickness is T*taper_factor.
        delta = L * taper_ratio # length of the taper region.

        # Determine grid extent using the larger of L and T.
        grid_extent = int(np.ceil(np.max([L, T]) / np.min(voxel_size[:2])))

        # Create a 2D grid.
        Y, X = np.meshgrid(
            np.arange(-grid_extent, grid_extent) * voxel_size[1],
            np.arange(-grid_extent, grid_extent) * voxel_size[0],
        )

        # Rotate the grid by the negative rotation angle.
        if rotation != 0:
            Xt = X * np.cos(rotation) - Y * np.sin(rotation)
            Yt = -X * np.sin(rotation) + Y * np.cos(rotation)
            X, Y = Xt, Yt

        absX = np.abs(X)

        T_eff = np.where(
            absX <= (L - delta),
            T,
            np.where(
                absX < L,
                T - (absX - (L - delta)) / delta * (T - T * taper_factor), 0)
            )

        # Mask is 1 where |x| < L and |y| is within the effective half‑thickness.
        mask = ((absX < L) & (np.abs(Y) <= T_eff)).astype(float)
        mask = np.expand_dims(mask, axis=-1)
        
        return mask


## Brightfield 
In `DeepTrack2`, the current brightfield microscope model simulates illumination using a single wavelength of light. This is equivalent to a brightfield microscope operating with a monochromatic light source.

In experimental setups however, a brightfield microscope typically uses a broad spectrum of wavelengths (white light).

To achieve a more realistic simulation of white light, we will need to simulate multiple brightfield images (sampling different parts of the visible light spectrum) and then averaging the contributions of these images.

We start by generating a spectrum of wavelengths and a bacterium scatterer.

In [ ]:
IMAGE_SIZE = 200

# Generate a spectrum of wavelengths to sample from, 5 will be enough.
spectrum = np.linspace(450e-9, 700e-9, 5)
bacterium = Rod(
    radius=(0.1e-6, .6e-6),
    position= (IMAGE_SIZE//2, IMAGE_SIZE//2)
)


In [ ]:
# Sample the wavelengths.
imaged_particle_list = []
for wavelength in spectrum:
    # Create a brightfield microscope for a given wavelength.
    single_wavelength_optics = dt.Brightfield(
        NA=1.4,
        resolution=1e-6,
        magnification=15,
        wavelength=wavelength,
        padding=(32, 32, 32, 32),
        output_region=(0, 0, IMAGE_SIZE, IMAGE_SIZE),
    )
    # Image the particle.
    imaged_particle = single_wavelength_optics(bacterium)

    # Add background noise.
    imaged_particle = imaged_particle >> dt.Gaussian(-2, 0.01)

    # Append to list.
    imaged_particle_list.append(imaged_particle)

# Take the average of the images in the list.
averaged_image = (sum(imaged_particle_list) / len(imaged_particle_list)).resolve()
plt.imshow(averaged_image, cmap="gray")

## Fluorescence
Fluorescence images in `DeepTrack2` are simulated with discrete volumes to act as light sources (fluorophores) which emit light from a scatterer.

Simulating fluorescence images are easy and straightforward to implement, we start by defining our microscope.

In [ ]:
# Create a fluorescence microscope for a given wavelength.
fluorescence_optics = dt.Fluorescence(
    NA=1.4,
    resolution=1e-6,
    magnification=10,
    wavelength=600e-9,
    padding=(32, 32, 32, 32),
    output_region=(0, 0, IMAGE_SIZE, IMAGE_SIZE),
)

# Image the particle.
imaged_particle = fluorescence_optics(bacterium)

# Add background noise.
imaged_particle_fluorescence = (imaged_particle >> dt.Gaussian(0, 0.005)).resolve()

plt.imshow(imaged_particle_fluorescence, cmap="gray")

## (Brightfield) Simulating active Brownian motion
We can combine the optical tools together with equations of motion to simulate a realistic movie of a particle diffusing with Brownian motion.

In [ ]:
# Make image bigger.
IMAGE_SIZE = 200

simulated_movie = []
movie_length = 40
dx, dy, dphi = 0, 0, 0
drift_velocity = 1.1

for t in range(movie_length):
    
    # Sample the wavelengths.
    imaged_particle_list = []
    for wavelength in spectrum:

        # Create a brightfield microscope for a given wavelength.
        single_wavelength_optics = dt.Brightfield(
            NA=1.4,
            resolution=1e-6,
            magnification=15,
            wavelength=wavelength,
            padding=(32, 32, 32, 32),
            output_region=(0, 0, IMAGE_SIZE, IMAGE_SIZE),
        )
        bacterium = Rod(radius=(0.07e-6, .6e-6),
          rotation=( dphi),
          position=(IMAGE_SIZE//2 + dx, IMAGE_SIZE//2 + dy),)
        
        # Image the bacterium.
        imaged_particle = single_wavelength_optics(bacterium)

        # Add background noise.
        imaged_particle = imaged_particle >> dt.Gaussian(-2, 0.01)

        # Append to list.
        imaged_particle_list.append(imaged_particle)

    # Take the average of the images in the list.
    averaged_image = (sum(imaged_particle_list) / len(imaged_particle_list)).resolve()

    simulated_movie.append(averaged_image)

    # Update displacements.
    dphi += np.random.standard_normal()*np.pi/50
    dx += np.random.standard_normal()*0.6 - drift_velocity*np.cos(dphi)
    dy += np.random.standard_normal()*0.6 - drift_velocity*np.sin(dphi)


plt.imshow(simulated_movie[0], cmap="gray")

## Make an animation from the simulation.
Using `FuncAnimation` and a `HTML` player, we will combine the simulated frames into a movie.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
def update(frame):
    display_frame = simulated_movie[frame]

    ax.clear()
    ax.set_axis_off()
    ax.imshow(display_frame, cmap="gray")
    ax.set_xlim(0, display_frame.shape[1])
    ax.set_ylim(display_frame.shape[0], 0)
    ax.set_title("Active Brownian Motion with Brightfield")

    return ax

# Make and display animation.
animation = FuncAnimation(
    fig,
    update,
    frames=len(simulated_movie)
)
player = HTML(animation.to_jshtml(fps=10)); plt.close()
player

## (Fluorescence) Simulating active Brownian motion
We can combine the optical tools together with equations of motion to simulate a realistic movie of a particle diffusing with Brownian motion.

In [ ]:
# Make image bigger.
IMAGE_SIZE = 150

simulated_movie = []
movie_length = 40
dx, dy, dphi = 0, 0, 0
drift_velocity = 1.1

for t in range(movie_length):
    imaged_particle_list = []
    bacterium = Rod(radius=(0.1e-6, .6e-6),
          rotation=( dphi),
          position=(IMAGE_SIZE//2 + dx, IMAGE_SIZE//2 + dy),)

    image = fluorescence_optics(bacterium)

    # Add background noise.
    image = image >> dt.Gaussian(0, 0.005)

    simulated_movie.append(image.resolve())

    dphi += np.random.standard_normal()*np.pi/50
    dx += np.random.standard_normal()*0.6 - drift_velocity*np.cos(dphi)
    dy += np.random.standard_normal()*0.6 - drift_velocity*np.sin(dphi)

plt.imshow(simulated_movie[0],cmap="gray")


## Make an animation from the simulation
Using `FuncAnimation` and a `HTML` player, we will combine the simulated frames into a movie.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
def update(frame):
    display_frame = simulated_movie[frame]

    ax.clear()
    ax.set_axis_off()
    ax.imshow(display_frame, cmap="gray")
    ax.set_xlim(0, display_frame.shape[1])
    ax.set_ylim(display_frame.shape[0], 0)
    ax.set_title("Active Brownian Motion with Fluorescence")

    return ax

# Make and display animation.
animation = FuncAnimation(
    fig,
    update,
    frames=len(simulated_movie)
)
player = HTML(animation.to_jshtml(fps=10)); plt.close()
player